# MODULE 6 — Final Cache QA, Source-Only Normalization, and Baseline Dataset Interface

**Project:** Strict Cross-Dataset, Subject-Independent Motor-Imagery EEG Classification

## Purpose

Module 5 v2 produced the publication-facing unnormalized EEG cache:

- 22 common EEG channels
- 160 Hz
- 8–30 Hz
- 0–4 s post-event
- 640 samples/epoch
- 118 subjects
- 9,316 epochs

Module 6 now creates the **reproducible data interface** used by all later
experiments.

This module performs:

1. independent HDF5 cache integrity verification;
2. class/subject/dataset/run consistency checks;
3. duplicate trial detection;
4. post-preprocessing class-balance verification;
5. deterministic subject-level fold generation;
6. source-only normalization fitting;
7. target-safe transformation;
8. baseline-ready PyTorch Dataset/DataLoader utilities;
9. fold manifests for classical and deep-learning baselines;
10. machine-readable protocol locking.

## Strict evaluation rule

For each held-out target subject:

- target epochs are never used to fit normalization;
- target epochs are never used for feature selection;
- target epochs are never used for threshold selection;
- target epochs are never used to choose hyperparameters;
- target epochs are only loaded for final evaluation.

No model is trained in Module 6.

## Primary evaluation interfaces

The module supports two fold types:

### A. Within-dataset subject-level LOSO

For each dataset separately:

`all subjects except target -> source training`

`target subject -> final test`

### B. Cross-dataset zero-calibration transfer

For:

- BCI-IV-2a → EEGMMIDB
- EEGMMIDB → BCI-IV-2a

the entire source dataset is available for training and every target subject is
held out for final evaluation.

These fold definitions are saved, but model training happens later.

## Source-only normalization

The primary normalization is:

\[
z_{c} = rac{x_c - \mathrm{median}_{source}}{\mathrm{IQR}_{source}+\epsilon}
\]

Statistics are fitted on **source training epochs only**.

The target subject receives exactly those frozen source statistics.

There is deliberately no:
- target centering,
- target variance normalization,
- target batch-statistics fitting,
- transductive normalization.

## Cell 1 — Imports and frozen Module 5 artifacts

In [1]:
# ============================================================
# CELL 1 — IMPORTS + FROZEN MODULE 5 ARTIFACTS
# ============================================================

from __future__ import annotations

import json
import hashlib
import random
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional, Iterable

import numpy as np
import pandas as pd
import h5py
import torch
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
)

PROJECT_ROOT = Path(
    "/Users/ashokvarmabevara/Project2"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "cross_dataset_mi_project"
)

MANIFEST_ROOT = (
    OUTPUT_ROOT
    / "manifests"
)

CACHE_ROOT = (
    OUTPUT_ROOT
    / "cache"
)

RESULTS_ROOT = (
    OUTPUT_ROOT
    / "results"
)

CONFIG_ROOT = (
    OUTPUT_ROOT
    / "config"
)

for p in [
    MANIFEST_ROOT,
    CACHE_ROOT,
    RESULTS_ROOT,
    CONFIG_ROOT,
]:
    p.mkdir(
        parents=True,
        exist_ok=True,
    )

FINAL_CACHE_PATH = (
    CACHE_ROOT
    / "module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5"
)

MODULE_5_SPEC_PATH = (
    MANIFEST_ROOT
    / "module_5_v2_preprocessing_specification.json"
)

MODULE_5_BALANCE_PATH = (
    MANIFEST_ROOT
    / "module_5_v2_post_preprocessing_balance.csv"
)

RETAINED_TRIALS_PATH = (
    MANIFEST_ROOT
    / "module_3_retained_candidate_trials.csv"
)

assert FINAL_CACHE_PATH.exists(), (
    f"Missing Module 5 cache: {FINAL_CACHE_PATH}"
)

assert MODULE_5_SPEC_PATH.exists(), (
    f"Missing Module 5 specification: {MODULE_5_SPEC_PATH}"
)

print("=" * 78)
print("MODULE 6 — FINAL CACHE QA + NORMALIZATION + DATA INTERFACE")
print("=" * 78)

print("Cache:", FINAL_CACHE_PATH)
print("Module 5 specification:", MODULE_5_SPEC_PATH)

MODULE 6 — FINAL CACHE QA + NORMALIZATION + DATA INTERFACE
Cache: /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/cache/module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5
Module 5 specification: /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/module_5_v2_preprocessing_specification.json


## Cell 2 — Load and validate the frozen preprocessing contract

In [2]:
# ============================================================
# CELL 2 — LOAD MODULE 5 SPECIFICATION
# ============================================================

with open(
    MODULE_5_SPEC_PATH,
    "r",
    encoding="utf-8",
) as f:
    module_5_spec = json.load(f)

FROZEN_CHANNELS = list(
    module_5_spec["channels"]
)

TARGET_SFREQ = float(
    module_5_spec["target_sfreq_hz"]
)

LOW_HZ, HIGH_HZ = map(
    float,
    module_5_spec["primary_bandpass_hz"]
)

TMIN = float(
    module_5_spec["epoch"]["tmin_sec"]
)

TMAX = float(
    module_5_spec["epoch"]["tmax_sec"]
)

N_SAMPLES = int(
    module_5_spec["epoch"]["n_samples"]
)

assert len(FROZEN_CHANNELS) == 22
assert TARGET_SFREQ == 160.0
assert LOW_HZ == 8.0
assert HIGH_HZ == 30.0
assert TMAX - TMIN == 4.0
assert N_SAMPLES == 640

print(
    json.dumps(
        {
            "n_channels": len(FROZEN_CHANNELS),
            "channels": FROZEN_CHANNELS,
            "target_sfreq_hz": TARGET_SFREQ,
            "bandpass_hz": [LOW_HZ, HIGH_HZ],
            "epoch_sec": [TMIN, TMAX],
            "n_samples": N_SAMPLES,
        },
        indent=2,
    )
)

print("\nModule 5 contract validation: PASS")

{
  "n_channels": 22,
  "channels": [
    "Fz",
    "FC3",
    "FC1",
    "FCz",
    "FC2",
    "FC4",
    "C5",
    "C3",
    "C1",
    "Cz",
    "C2",
    "C4",
    "C6",
    "CP3",
    "CP1",
    "CPz",
    "CP2",
    "CP4",
    "P1",
    "Pz",
    "P2",
    "POz"
  ],
  "target_sfreq_hz": 160.0,
  "bandpass_hz": [
    8.0,
    30.0
  ],
  "epoch_sec": [
    0.0,
    4.0
  ],
  "n_samples": 640
}

Module 5 contract validation: PASS


## Cell 3 — Independent HDF5 cache inspection

This is deliberately independent from the Module 5 creation logic.

We verify:
- datasets exist;
- shape;
- dtype;
- metadata fields;
- cache attributes;
- finite values;
- normalized flag.

In [3]:
# ============================================================
# CELL 3 — INDEPENDENT HDF5 CACHE INSPECTION
# ============================================================

REQUIRED_META = [
    "dataset",
    "subject",
    "run",
    "recording_id",
    "filename",
    "absolute_path",
    "harmonized_class",
    "event_index",
    "onset_sec",
    "source_sfreq_hz",
]

with h5py.File(
    FINAL_CACHE_PATH,
    "r",
) as h5:

    assert "X" in h5, (
        "HDF5 cache does not contain dataset X."
    )

    X_shape = tuple(
        h5["X"].shape
    )

    X_dtype = str(
        h5["X"].dtype
    )

    assert "metadata" in h5, (
        "HDF5 cache does not contain metadata group."
    )

    metadata_keys = list(
        h5["metadata"].keys()
    )

    missing_meta = [
        k
        for k in REQUIRED_META
        if k not in metadata_keys
    ]

    cache_attrs = {
        key: h5.attrs[key]
        for key in h5.attrs.keys()
    }

    print("=" * 78)
    print("HDF5 CACHE INSPECTION")
    print("=" * 78)

    print("Shape:", X_shape)
    print("Dtype:", X_dtype)

    print(
        "Metadata keys:",
        metadata_keys,
    )

    print(
        "Missing required metadata:",
        missing_meta,
    )

    print("\nCache attributes:")
    for k, v in cache_attrs.items():
        print(f"  {k}: {v}")

assert X_shape[1:] == (
    22,
    640,
)

assert X_dtype == "float32"

assert len(
    missing_meta
) == 0

assert cache_attrs.get(
    "normalized",
    None,
) in [
    False,
    np.bool_(False),
]

print("\nBasic HDF5 structure: PASS")

HDF5 CACHE INSPECTION
Shape: (9316, 22, 640)
Dtype: float32
Metadata keys: ['absolute_path', 'dataset', 'event_index', 'filename', 'harmonized_class', 'onset_sec', 'recording_id', 'run', 'source_sfreq_hz', 'subject']
Missing required metadata: []

Cache attributes:
  bandpass_high_hz: 30.0
  bandpass_low_hz: 8.0
  continuous_preprocessing: True
  epoch_tmax_sec: 4.0
  epoch_tmin_sec: 0.0
  filtering: MNE FIR zero-phase on continuous 160-Hz recording
  interpolation: False
  module: 5
  n_channels: 22
  n_samples: 640
  normalized: False
  reference: average reference over 22 common EEG channels
  resampling: scipy.signal.resample_poly on continuous recording
  target_sfreq_hz: 160.0
  version: v2_refined

Basic HDF5 structure: PASS


## Cell 4 — Load metadata into a DataFrame

The signal matrix stays in HDF5.

Metadata are loaded into memory because they are small and are needed for:
- subject-level splitting;
- class verification;
- fold creation.

In [4]:
# ============================================================
# CELL 4 — LOAD CACHE METADATA
# ============================================================

def decode_h5_strings(values):
    out = []

    for value in values:
        if isinstance(value, bytes):
            out.append(
                value.decode(
                    "utf-8"
                )
            )
        else:
            out.append(
                str(value)
            )

    return out


with h5py.File(
    FINAL_CACHE_PATH,
    "r",
) as h5:

    n_epochs = int(
        h5["X"].shape[0]
    )

    metadata = {}

    for key in [
        "dataset",
        "subject",
        "run",
        "recording_id",
        "filename",
        "absolute_path",
        "harmonized_class",
    ]:
        metadata[key] = decode_h5_strings(
            h5["metadata"][key][:]
        )

    metadata["event_index"] = (
        h5["metadata"]["event_index"][:]
        .astype(np.int64)
    )

    metadata["onset_sec"] = (
        h5["metadata"]["onset_sec"][:]
        .astype(np.float64)
    )

    metadata["source_sfreq_hz"] = (
        h5["metadata"]["source_sfreq_hz"][:]
        .astype(np.float64)
    )

cache_meta_df = pd.DataFrame(
    metadata
)

cache_meta_df.insert(
    0,
    "cache_index",
    np.arange(
        n_epochs,
        dtype=np.int64,
    ),
)

print(
    "Loaded metadata rows:",
    len(cache_meta_df),
)

display(
    cache_meta_df.head(10)
)

assert len(cache_meta_df) == X_shape[0]

Loaded metadata rows: 9316


,cache_index,dataset,subject,run,recording_id,filename,absolute_path,harmonized_class,event_index,onset_sec,source_sfreq_hz
0,0,BCI-IV-2a,S01,T,A01T,A01T.gdf,/Users/ashokvarmabevara/Project2/BCI IV-2a/A01...,feet,10,377.484,250.0
1,1,BCI-IV-2a,S01,T,A01T,A01T.gdf,/Users/ashokvarmabevara/Project2/BCI IV-2a/A01...,right,12,385.156,250.0
2,2,BCI-IV-2a,S01,T,A01T,A01T.gdf,/Users/ashokvarmabevara/Project2/BCI IV-2a/A01...,left,14,392.964,250.0
3,3,BCI-IV-2a,S01,T,A01T,A01T.gdf,/Users/ashokvarmabevara/Project2/BCI IV-2a/A01...,left,16,400.996,250.0
4,4,BCI-IV-2a,S01,T,A01T,A01T.gdf,/Users/ashokvarmabevara/Project2/BCI IV-2a/A01...,right,18,409.440,250.0
5,5,BCI-IV-2a,S01,T,A01T,A01T.gdf,/Users/ashokvarmabevara/Project2/BCI IV-2a/A01...,feet,20,417.108,250.0
6,6,BCI-IV-2a,S01,T,A01T,A01T.gdf,/Users/ashokvarmabevara/Project2/BCI IV-2a/A01...,right,24,433.032,250.0
7,7,BCI-IV-2a,S01,T,A01T,A01T.gdf,/Users/ashokvarmabevara/Project2/BCI IV-2a/A01...,feet,26,441.024,250.0
8,8,BCI-IV-2a,S01,T,A01T,A01T.gdf,/Users/ashokvarmabevara/Project2/BCI IV-2a/A01...,left,28,448.648,250.0
9,9,BCI-IV-2a,S01,T,A01T,A01T.gdf,/Users/ashokvarmabevara/Project2/BCI IV-2a/A01...,left,30,456.228,250.0


## Cell 5 — Final cache QA: uniqueness and referential integrity

Each epoch must be uniquely identified by:

`dataset + subject + recording_id + event_index`

We also verify that:
- event index is non-negative;
- onset is finite;
- source sampling rate is one of 128/160/250 Hz;
- labels are exactly Left/Right/Feet.

In [5]:
# ============================================================
# CELL 5 — CACHE REFERENTIAL INTEGRITY
# ============================================================

TRIAL_KEY = [
    "dataset",
    "subject",
    "recording_id",
    "event_index",
]

duplicate_trial_keys = (
    cache_meta_df[
        cache_meta_df.duplicated(
            subset=TRIAL_KEY,
            keep=False,
        )
    ]
    .sort_values(TRIAL_KEY)
)

print("=" * 78)
print("CACHE REFERENTIAL INTEGRITY")
print("=" * 78)

print(
    "Duplicate trial keys:",
    len(duplicate_trial_keys),
)

if len(duplicate_trial_keys):
    display(
        duplicate_trial_keys.head(50)
    )

assert len(
    duplicate_trial_keys
) == 0

assert (
    cache_meta_df["event_index"] >= 0
).all()

assert (
    np.isfinite(
        cache_meta_df["onset_sec"].to_numpy()
    ).all()
)

assert set(
    cache_meta_df["harmonized_class"].unique()
).issubset(
    {
        "left",
        "right",
        "feet",
    }
)

assert set(
    cache_meta_df["source_sfreq_hz"].unique()
).issubset(
    {
        128.0,
        160.0,
        250.0,
    }
)

print("Trial uniqueness: PASS")
print("Event metadata: PASS")
print("Label integrity: PASS")
print("Source-rate metadata: PASS")

CACHE REFERENTIAL INTEGRITY
Duplicate trial keys: 0
Trial uniqueness: PASS
Event metadata: PASS
Label integrity: PASS
Source-rate metadata: PASS


## Cell 6 — Full finite-value scan and signal sanity checks

The cache should contain no NaNs or Infs.

We also calculate:
- global mean;
- global standard deviation;
- per-channel standard deviation;
- zero-variance channels.

These statistics are QA only. They are not used to normalize the cache.

In [6]:
# ============================================================
# CELL 6 — FULL CACHE NUMERICAL QA
# ============================================================

nonfinite_count = 0
zero_variance_epoch_count = 0
global_sum = 0.0
global_sq_sum = 0.0
global_n = 0

channel_sum = np.zeros(
    22,
    dtype=np.float64,
)

channel_sq_sum = np.zeros(
    22,
    dtype=np.float64,
)

with h5py.File(
    FINAL_CACHE_PATH,
    "r",
) as h5:

    X = h5["X"]

    for start in range(
        0,
        X.shape[0],
        64,
    ):

        stop = min(
            start + 64,
            X.shape[0],
        )

        block = np.asarray(
            X[start:stop],
            dtype=np.float64,
        )

        nonfinite_count += int(
            (~np.isfinite(block)).sum()
        )

        channel_var = np.var(
            block,
            axis=2,
        )

        zero_variance_epoch_count += int(
            (
                channel_var
                <= np.finfo(
                    np.float64
                ).eps
            ).any(
                axis=1
            ).sum()
        )

        global_sum += float(
            block.sum()
        )

        global_sq_sum += float(
            (block ** 2).sum()
        )

        global_n += block.size

        channel_sum += block.sum(
            axis=(0, 2)
        )

        channel_sq_sum += (
            block ** 2
        ).sum(
            axis=(0, 2)
        )

global_mean = (
    global_sum
    / global_n
)

global_var = (
    global_sq_sum
    / global_n
    - global_mean ** 2
)

global_std = float(
    np.sqrt(
        max(
            global_var,
            0.0,
        )
    )
)

channel_mean = (
    channel_sum
    / (
        X_shape[0]
        * X_shape[2]
    )
)

channel_var = (
    channel_sq_sum
    / (
        X_shape[0]
        * X_shape[2]
    )
    - channel_mean ** 2
)

channel_std = np.sqrt(
    np.maximum(
        channel_var,
        0.0,
    )
)

print("=" * 78)
print("NUMERICAL CACHE QA")
print("=" * 78)

print(
    "Non-finite values:",
    nonfinite_count,
)

print(
    "Epochs with zero-variance channel:",
    zero_variance_epoch_count,
)

print(
    "Global mean:",
    global_mean,
)

print(
    "Global std:",
    global_std,
)

print("\nPer-channel standard deviations:")
display(
    pd.DataFrame({
        "channel": FROZEN_CHANNELS,
        "std": channel_std,
    })
)

assert nonfinite_count == 0
assert zero_variance_epoch_count == 0
assert np.isfinite(global_std)
assert global_std > 0
assert np.all(
    channel_std > 0
)

print("\nFull numerical cache QA: PASS")

NUMERICAL CACHE QA
Non-finite values: 0
Epochs with zero-variance channel: 0
Global mean: -9.127994707785321e-18
Global std: 9.757151288904615e-06

Per-channel standard deviations:


,channel,std
0,Fz,0.000010
1,FC3,0.000010
2,FC1,0.000008
3,FCz,0.000008
4,FC2,0.000008
5,FC4,0.000010
6,C5,0.000013
7,C3,0.000008
8,C1,0.000006
9,Cz,0.000006



Full numerical cache QA: PASS


## Cell 7 — Reproduce final class and subject statistics

This independently confirms that the cache still contains all three classes and
all 118 subjects.

In [7]:
# ============================================================
# CELL 7 — FINAL DATASET BALANCE
# ============================================================

class_counts = (
    cache_meta_df
    .groupby(
        [
            "dataset",
            "harmonized_class",
        ]
    )
    .size()
    .reset_index(
        name="epochs"
    )
)

subject_counts = (
    cache_meta_df
    .groupby(
        [
            "dataset",
            "subject",
        ]
    )
    .size()
    .reset_index(
        name="epochs"
    )
)

print("=" * 78)
print("FINAL CACHE BALANCE")
print("=" * 78)

print("Class counts:")
display(
    class_counts
)

print("\nSubject counts summary:")
display(
    subject_counts
    .groupby("dataset")[
        "epochs"
    ]
    .agg(
        subjects="count",
        min="min",
        median="median",
        mean="mean",
        max="max",
    )
    .reset_index()
)

total_subjects = (
    cache_meta_df[
        "subject"
    ].nunique()
)

print(
    "\nTotal unique subjects:",
    total_subjects,
)

assert total_subjects == 118

for dataset_name, group in (
    cache_meta_df.groupby("dataset")
):
    class_set = set(
        group[
            "harmonized_class"
        ].unique()
    )

    assert class_set == {
        "left",
        "right",
        "feet",
    }

print(
    "\nClass/subject balance QA: PASS"
)

FINAL CACHE BALANCE
Class counts:


,dataset,harmonized_class,epochs
0,BCI-IV-2a,feet,648
1,BCI-IV-2a,left,648
2,BCI-IV-2a,right,648
3,EEGMMIDB,feet,2455
4,EEGMMIDB,left,2479
5,EEGMMIDB,right,2438



Subject counts summary:


,dataset,subjects,min,median,mean,max
0,BCI-IV-2a,9,216,216.0,216.000000,216
1,EEGMMIDB,109,54,68.0,67.633028,85



Total unique subjects: 118

Class/subject balance QA: PASS


## Cell 8 — Source-only normalizer

The normalizer is implemented as a reusable object.

It never modifies the HDF5 cache.

Its fitted state contains only source training statistics.

In [8]:
# ============================================================
# CELL 8 — SOURCE-ONLY ROBUST NORMALIZER
# ============================================================

class SourceOnlyRobustNormalizer:
    """
    Leakage-safe channel-wise robust normalization.

    Fit:
        source training epochs only.

    Transform:
        source validation, source test, or target test
        using frozen source statistics.

    No target statistics are computed.
    """

    def __init__(
        self,
        eps: float = 1e-6,
    ):
        self.eps = float(eps)
        self.median_ = None
        self.iqr_ = None
        self.fitted_subjects_ = None
        self.fitted_ = False

    def fit(
        self,
        X_source: np.ndarray,
        source_subjects: Iterable[str],
    ):
        X_source = np.asarray(
            X_source,
            dtype=np.float64,
        )

        source_subjects = [
            str(s)
            for s in source_subjects
        ]

        if X_source.ndim != 3:
            raise ValueError(
                "X_source must have shape (N,C,T)."
            )

        if X_source.shape[1:] != (
            22,
            640,
        ):
            raise ValueError(
                f"Unexpected source shape: "
                f"{X_source.shape}"
            )

        if len(source_subjects) != X_source.shape[0]:
            raise ValueError(
                "One subject id is required per epoch."
            )

        if len(
            set(source_subjects)
        ) == 0:
            raise ValueError(
                "No source subjects supplied."
            )

        if not np.isfinite(
            X_source
        ).all():
            raise ValueError(
                "Source data contains non-finite values."
            )

        values = (
            X_source
            .transpose(
                1,
                0,
                2,
            )
            .reshape(
                22,
                -1,
            )
        )

        self.median_ = np.median(
            values,
            axis=1,
        )

        q25 = np.percentile(
            values,
            25,
            axis=1,
        )

        q75 = np.percentile(
            values,
            75,
            axis=1,
        )

        self.iqr_ = np.maximum(
            q75 - q25,
            self.eps,
        )

        self.fitted_subjects_ = tuple(
            sorted(
                set(source_subjects)
            )
        )

        self.fitted_ = True

        return self

    def transform(
        self,
        X: np.ndarray,
    ) -> np.ndarray:

        if not self.fitted_:
            raise RuntimeError(
                "Normalizer has not been fitted."
            )

        X = np.asarray(
            X,
            dtype=np.float64,
        )

        if X.ndim != 3:
            raise ValueError(
                "X must have shape (N,C,T)."
            )

        if X.shape[1:] != (
            22,
            640,
        ):
            raise ValueError(
                f"Unexpected transform shape: {X.shape}"
            )

        return (
            X
            - self.median_[None, :, None]
        ) / (
            self.iqr_[None, :, None]
            + self.eps
        )

    def assert_target_excluded(
        self,
        target_subject: str,
    ):
        if not self.fitted_:
            raise RuntimeError(
                "Normalizer is not fitted."
            )

        target_subject = str(
            target_subject
        )

        if target_subject in set(
            self.fitted_subjects_
        ):
            raise AssertionError(
                f"LEAKAGE: target subject "
                f"{target_subject} was included "
                "in normalization fitting."
            )

        return True


print("Source-only normalizer definition: PASS")

Source-only normalizer definition: PASS


## Cell 9 — HDF5 indexed loader

This interface lets later experiments read only selected epoch indices rather
than loading the entire cache.

That is important for memory efficiency on the MacBook Air M4.

In [9]:
# ============================================================
# CELL 9 — INDEXED HDF5 LOADER
# ============================================================

class HDF5EpochStore:
    """
    Lightweight random-access reader for the final cache.
    """

    def __init__(
        self,
        path: Path,
    ):
        self.path = Path(path)
        self._handle = None

    def open(self):
        if self._handle is None:
            self._handle = h5py.File(
                self.path,
                "r",
            )
        return self

    def close(self):
        if self._handle is not None:
            self._handle.close()
            self._handle = None

    def __enter__(self):
        return self.open()

    def __exit__(
        self,
        exc_type,
        exc,
        tb,
    ):
        self.close()

    @property
    def shape(self):
        self.open()
        return tuple(
            self._handle["X"].shape
        )

    def get_X(
        self,
        indices,
    ):
        self.open()
        indices = np.asarray(
            indices,
            dtype=np.int64,
        )

        # HDF5 fancy indexing requires sorted indices in many h5py versions.
        order = np.argsort(indices)
        sorted_idx = indices[order]

        values = np.asarray(
            self._handle["X"][
                sorted_idx
            ]
        )

        inverse = np.argsort(
            order
        )

        return values[
            inverse
        ]

    def get_metadata(
        self,
        indices,
    ):
        self.open()

        indices = np.asarray(
            indices,
            dtype=np.int64,
        )

        order = np.argsort(indices)
        sorted_idx = indices[order]
        inverse = np.argsort(order)

        result = {}

        for key in [
            "dataset",
            "subject",
            "run",
            "recording_id",
            "filename",
            "absolute_path",
            "harmonized_class",
        ]:

            values = decode_h5_strings(
                self._handle[
                    "metadata"
                ][key][sorted_idx]
            )

            result[key] = [
                values[i]
                for i in inverse
            ]

        result["event_index"] = (
            self._handle[
                "metadata"
            ]["event_index"][
                sorted_idx
            ][inverse]
        )

        result["onset_sec"] = (
            self._handle[
                "metadata"
            ]["onset_sec"][
                sorted_idx
            ][inverse]
        )

        result["source_sfreq_hz"] = (
            self._handle[
                "metadata"
            ]["source_sfreq_hz"][
                sorted_idx
            ][inverse]
        )

        return pd.DataFrame(
            result,
            index=indices,
        )


store = HDF5EpochStore(
    FINAL_CACHE_PATH
)

with store:
    assert store.shape == X_shape

    test_idx = np.array(
        [
            0,
            min(1, X_shape[0] - 1),
            X_shape[0] - 1,
        ],
        dtype=np.int64,
    )

    X_test = store.get_X(
        test_idx
    )

    meta_test = store.get_metadata(
        test_idx
    )

print(
    "Indexed X shape:",
    X_test.shape,
)

display(meta_test)

assert X_test.shape == (
    len(test_idx),
    22,
    640,
)

print("\nIndexed HDF5 interface: PASS")

Indexed X shape: (3, 22, 640)


,dataset,subject,run,recording_id,filename,absolute_path,harmonized_class,event_index,onset_sec,source_sfreq_hz
0,BCI-IV-2a,S01,T,A01T,A01T.gdf,/Users/ashokvarmabevara/Project2/BCI IV-2a/A01...,feet,10,377.484,250.0
1,BCI-IV-2a,S01,T,A01T,A01T.gdf,/Users/ashokvarmabevara/Project2/BCI IV-2a/A01...,right,12,385.156,250.0
9315,EEGMMIDB,S109,R14,S109R14,S109R14.edf,/Users/ashokvarmabevara/Project2/eegmmidb/S109...,feet,29,118.900,160.0



Indexed HDF5 interface: PASS


## Cell 10 — Deterministic within-dataset LOSO manifests

Each target subject is a separate held-out fold.

No target data are used to fit normalization or select hyperparameters.

The fold manifest contains only indices and subject IDs.

In [10]:
# ============================================================
# CELL 10 — WITHIN-DATASET LOSO FOLDS
# ============================================================

GLOBAL_SEED = 20260822

def build_within_dataset_loso(
    meta_df: pd.DataFrame,
    dataset_name: str,
) -> pd.DataFrame:

    sub = meta_df[
        meta_df["dataset"]
        == dataset_name
    ].copy()

    subjects = sorted(
        sub["subject"].unique()
    )

    rows = []

    for fold_id, target_subject in enumerate(
        subjects
    ):

        train_mask = (
            sub["subject"]
            != target_subject
        )

        test_mask = (
            sub["subject"]
            == target_subject
        )

        train_indices = (
            sub.loc[
                train_mask,
                "cache_index",
            ]
            .astype(int)
            .tolist()
        )

        test_indices = (
            sub.loc[
                test_mask,
                "cache_index",
            ]
            .astype(int)
            .tolist()
        )

        rows.append({
            "protocol": "within_dataset_loso",
            "dataset": dataset_name,
            "fold_id": int(fold_id),
            "target_subject": target_subject,
            "source_train_subject_count": len(
                set(
                    sub.loc[
                        train_mask,
                        "subject",
                    ]
                )
            ),
            "target_test_subject_count": 1,
            "source_train_epoch_count": len(
                train_indices
            ),
            "target_test_epoch_count": len(
                test_indices
            ),
            "train_indices_json": json.dumps(
                train_indices
            ),
            "test_indices_json": json.dumps(
                test_indices
            ),
        })

    return pd.DataFrame(
        rows
    )


loso_bci_df = build_within_dataset_loso(
    cache_meta_df,
    "BCI-IV-2a",
)

loso_phys_df = build_within_dataset_loso(
    cache_meta_df,
    "EEGMMIDB",
)

within_loso_df = pd.concat(
    [
        loso_bci_df,
        loso_phys_df,
    ],
    ignore_index=True,
)

print(
    "Within-dataset LOSO folds:",
    len(within_loso_df)
)

display(
    within_loso_df[
        [
            "dataset",
            "fold_id",
            "target_subject",
            "source_train_subject_count",
            "source_train_epoch_count",
            "target_test_epoch_count",
        ]
    ].head(20)
)

assert len(loso_bci_df) == 9
assert len(loso_phys_df) == 109

print("\nWithin-dataset LOSO manifest: PASS")

Within-dataset LOSO folds: 118


,dataset,fold_id,target_subject,source_train_subject_count,source_train_epoch_count,target_test_epoch_count
0,BCI-IV-2a,0,S01,8,1728,216
1,BCI-IV-2a,1,S02,8,1728,216
2,BCI-IV-2a,2,S03,8,1728,216
3,BCI-IV-2a,3,S04,8,1728,216
4,BCI-IV-2a,4,S05,8,1728,216
5,BCI-IV-2a,5,S06,8,1728,216
6,BCI-IV-2a,6,S07,8,1728,216
7,BCI-IV-2a,7,S08,8,1728,216
8,BCI-IV-2a,8,S09,8,1728,216
9,EEGMMIDB,0,S001,108,7303,69



Within-dataset LOSO manifest: PASS


## Cell 11 — Cross-dataset zero-calibration manifests

These are the key domain-generalization folds.

### Direction 1

`BCI-IV-2a → EEGMMIDB`

### Direction 2

`EEGMMIDB → BCI-IV-2a`

Every target subject is fully unseen during source training.

In [11]:
# ============================================================
# CELL 11 — CROSS-DATASET TRANSFER FOLDS
# ============================================================

def build_cross_dataset_transfer(
    meta_df: pd.DataFrame,
    source_dataset: str,
    target_dataset: str,
) -> pd.DataFrame:

    source = meta_df[
        meta_df["dataset"]
        == source_dataset
    ].copy()

    target = meta_df[
        meta_df["dataset"]
        == target_dataset
    ].copy()

    target_subjects = sorted(
        target["subject"].unique()
    )

    source_indices = (
        source["cache_index"]
        .astype(int)
        .tolist()
    )

    rows = []

    for fold_id, target_subject in enumerate(
        target_subjects
    ):

        test_indices = (
            target.loc[
                target["subject"]
                == target_subject,
                "cache_index",
            ]
            .astype(int)
            .tolist()
        )

        rows.append({
            "protocol": "cross_dataset_zero_calibration",
            "source_dataset": source_dataset,
            "target_dataset": target_dataset,
            "fold_id": int(fold_id),
            "target_subject": target_subject,
            "source_train_subject_count": int(
                source["subject"].nunique()
            ),
            "source_train_epoch_count": len(
                source_indices
            ),
            "target_test_epoch_count": len(
                test_indices
            ),
            "train_indices_json": json.dumps(
                source_indices
            ),
            "test_indices_json": json.dumps(
                test_indices
            ),
        })

    return pd.DataFrame(
        rows
    )


transfer_bci_to_phys = (
    build_cross_dataset_transfer(
        cache_meta_df,
        "BCI-IV-2a",
        "EEGMMIDB",
    )
)

transfer_phys_to_bci = (
    build_cross_dataset_transfer(
        cache_meta_df,
        "EEGMMIDB",
        "BCI-IV-2a",
    )
)

transfer_df = pd.concat(
    [
        transfer_bci_to_phys,
        transfer_phys_to_bci,
    ],
    ignore_index=True,
)

print(
    "Cross-dataset target folds:",
    len(transfer_df),
)

display(
    transfer_df[
        [
            "source_dataset",
            "target_dataset",
            "fold_id",
            "target_subject",
            "source_train_subject_count",
            "source_train_epoch_count",
            "target_test_epoch_count",
        ]
    ].head(20)
)

assert len(
    transfer_bci_to_phys
) == 109

assert len(
    transfer_phys_to_bci
) == 9

print(
    "\nCross-dataset zero-calibration manifest: PASS"
)

Cross-dataset target folds: 118


,source_dataset,target_dataset,fold_id,target_subject,source_train_subject_count,source_train_epoch_count,target_test_epoch_count
0,BCI-IV-2a,EEGMMIDB,0,S001,9,1944,69
1,BCI-IV-2a,EEGMMIDB,1,S002,9,1944,66
2,BCI-IV-2a,EEGMMIDB,2,S003,9,1944,69
3,BCI-IV-2a,EEGMMIDB,3,S004,9,1944,68
4,BCI-IV-2a,EEGMMIDB,4,S005,9,1944,67
5,BCI-IV-2a,EEGMMIDB,5,S006,9,1944,68
6,BCI-IV-2a,EEGMMIDB,6,S007,9,1944,68
7,BCI-IV-2a,EEGMMIDB,7,S008,9,1944,67
8,BCI-IV-2a,EEGMMIDB,8,S009,9,1944,67
9,BCI-IV-2a,EEGMMIDB,9,S010,9,1944,67



Cross-dataset zero-calibration manifest: PASS


## Cell 12 — Leakage audit for every fold

This is a critical validation step.

For each fold:
- target subject cannot appear in source training subjects;
- target indices cannot overlap source indices;
- target dataset cannot be used to fit source statistics.

This module performs the split-level checks now; the actual normalizer fit
will re-check them at runtime later.

In [12]:
# ============================================================
# CELL 12 — FOLD LEAKAGE AUDIT
# ============================================================

def parse_indices(
    value: str,
) -> np.ndarray:
    return np.asarray(
        json.loads(value),
        dtype=np.int64,
    )


def audit_within_loso(
    fold_df: pd.DataFrame,
    meta_df: pd.DataFrame,
):

    issues = []

    for _, fold in fold_df.iterrows():

        train_idx = parse_indices(
            fold["train_indices_json"]
        )

        test_idx = parse_indices(
            fold["test_indices_json"]
        )

        if len(
            np.intersect1d(
                train_idx,
                test_idx,
            )
        ):
            issues.append({
                "protocol": "within_dataset_loso",
                "dataset": fold["dataset"],
                "target_subject": fold["target_subject"],
                "issue": "train_test_index_overlap",
            })

        train_subjects = set(
            meta_df.loc[
                train_idx,
                "subject",
            ]
        )

        if fold[
            "target_subject"
        ] in train_subjects:

            issues.append({
                "protocol": "within_dataset_loso",
                "dataset": fold["dataset"],
                "target_subject": fold["target_subject"],
                "issue": "target_subject_in_train",
            })

    return issues


def audit_transfer(
    fold_df: pd.DataFrame,
    meta_df: pd.DataFrame,
):

    issues = []

    for _, fold in fold_df.iterrows():

        train_idx = parse_indices(
            fold["train_indices_json"]
        )

        test_idx = parse_indices(
            fold["test_indices_json"]
        )

        if len(
            np.intersect1d(
                train_idx,
                test_idx,
            )
        ):
            issues.append({
                "protocol": fold["protocol"],
                "source_dataset": fold["source_dataset"],
                "target_dataset": fold["target_dataset"],
                "target_subject": fold["target_subject"],
                "issue": "train_test_index_overlap",
            })

        train_subjects = set(
            meta_df.loc[
                train_idx,
                "subject",
            ]
        )

        if fold[
            "target_subject"
        ] in train_subjects:

            issues.append({
                "protocol": fold["protocol"],
                "source_dataset": fold["source_dataset"],
                "target_dataset": fold["target_dataset"],
                "target_subject": fold["target_subject"],
                "issue": "target_subject_in_train",
            })

    return issues


within_issues = (
    audit_within_loso(
        within_loso_df[
            within_loso_df["dataset"]
            == "BCI-IV-2a"
        ].reset_index(drop=True),
        cache_meta_df,
    )
    +
    audit_within_loso(
        within_loso_df[
            within_loso_df["dataset"]
            == "EEGMMIDB"
        ].reset_index(drop=True),
        cache_meta_df,
    )
)

transfer_issues = audit_transfer(
    transfer_df,
    cache_meta_df,
)

all_fold_issues = (
    within_issues
    + transfer_issues
)

print(
    "Total fold leakage issues:",
    len(all_fold_issues),
)

if all_fold_issues:
    display(
        pd.DataFrame(
            all_fold_issues
        )
    )

assert len(
    all_fold_issues
) == 0

print("\nFold leakage audit: PASS")

Total fold leakage issues: 0

Fold leakage audit: PASS


## Cell 13 — Fold-safe normalization smoke test

We build a tiny demonstration from one real LOSO fold.

The target subject is deliberately excluded from fitting.

The fitted statistics are then applied to:
- source training;
- target test.

The target is never used to recompute median/IQR.

In [13]:
# ============================================================
# CELL 13 — FOLD-SAFE NORMALIZATION SMOKE TEST
# ============================================================

demo_fold = within_loso_df[
    within_loso_df["dataset"] == "BCI-IV-2a"
].iloc[0]

demo_train_idx = parse_indices(
    demo_fold["train_indices_json"]
)

demo_test_idx = parse_indices(
    demo_fold["test_indices_json"]
)

with HDF5EpochStore(
    FINAL_CACHE_PATH
) as store:

    X_train_demo = store.get_X(
        demo_train_idx[:32]
    )

    X_test_demo = store.get_X(
        demo_test_idx
    )

train_subjects_demo = cache_meta_df.loc[
    demo_train_idx[:32],
    "subject",
].tolist()

test_subjects_demo = cache_meta_df.loc[
    demo_test_idx,
    "subject",
].tolist()

normalizer_demo = (
    SourceOnlyRobustNormalizer()
    .fit(
        X_train_demo,
        train_subjects_demo,
    )
)

normalizer_demo.assert_target_excluded(
    demo_fold["target_subject"]
)

X_train_norm = normalizer_demo.transform(
    X_train_demo
)

X_test_norm = normalizer_demo.transform(
    X_test_demo
)

print(
    "Target subject:",
    demo_fold["target_subject"],
)

print(
    "Subjects used in fit:",
    normalizer_demo.fitted_subjects_,
)

print(
    "Target subjects:",
    sorted(
        set(test_subjects_demo)
    )
)

print(
    "Normalized train shape:",
    X_train_norm.shape,
)

print(
    "Normalized target shape:",
    X_test_norm.shape,
)

assert (
    demo_fold["target_subject"]
    not in
    set(
        normalizer_demo.fitted_subjects_
    )
)

assert X_train_norm.shape == X_train_demo.shape
assert X_test_norm.shape == X_test_demo.shape
assert np.isfinite(
    X_train_norm
).all()
assert np.isfinite(
    X_test_norm
).all()

print(
    "\nFold-safe normalization smoke test: PASS"
)

Target subject: S01
Subjects used in fit: ('S02',)
Target subjects: ['S01']
Normalized train shape: (32, 22, 640)
Normalized target shape: (216, 22, 640)

Fold-safe normalization smoke test: PASS


## Cell 14 — Build baseline-ready PyTorch Dataset

The dataset interface supports:
- raw unnormalized epochs;
- normalized epochs using a passed frozen normalizer;
- metadata;
- integer class labels.

No hidden normalization is performed inside the Dataset.

In [14]:
# ============================================================
# CELL 14 — BASELINE PYTORCH DATASET
# ============================================================

CLASS_TO_ID = {
    "left": 0,
    "right": 1,
    "feet": 2,
}

ID_TO_CLASS = {
    value: key
    for key, value in CLASS_TO_ID.items()
}


class EEGHDF5Dataset(Dataset):

    def __init__(
        self,
        cache_path: Path,
        indices,
        normalizer: Optional[
            SourceOnlyRobustNormalizer
        ] = None,
        return_metadata: bool = False,
        dtype=torch.float32,
    ):

        self.cache_path = Path(
            cache_path
        )

        self.indices = np.asarray(
            indices,
            dtype=np.int64,
        )

        self.normalizer = normalizer

        self.return_metadata = (
            bool(return_metadata)
        )

        self.dtype = dtype

        self._h5 = None

    def _ensure_open(self):

        if self._h5 is None:
            self._h5 = h5py.File(
                self.cache_path,
                "r",
            )

    def __len__(self):
        return len(
            self.indices
        )

    def __getitem__(
        self,
        item,
    ):

        self._ensure_open()

        cache_index = int(
            self.indices[item]
        )

        x = np.asarray(
            self._h5["X"][
                cache_index
            ],
            dtype=np.float32,
        )

        if self.normalizer is not None:

            x = self.normalizer.transform(
                x[
                    None,
                    ...
                ]
            )[0].astype(
                np.float32
            )

        label_name = (
            self._h5[
                "metadata"
            ]["harmonized_class"][
                cache_index
            ]
        )

        if isinstance(
            label_name,
            bytes,
        ):
            label_name = (
                label_name.decode(
                    "utf-8"
                )
            )

        label = CLASS_TO_ID[
            str(label_name)
        ]

        x_tensor = torch.from_numpy(
            x
        ).to(
            self.dtype
        )

        y_tensor = torch.tensor(
            label,
            dtype=torch.long,
        )

        if not self.return_metadata:
            return (
                x_tensor,
                y_tensor,
            )

        subject = self._h5[
            "metadata"
        ]["subject"][
            cache_index
        ]

        if isinstance(
            subject,
            bytes,
        ):
            subject = subject.decode(
                "utf-8"
            )

        return (
            x_tensor,
            y_tensor,
            {
                "cache_index": cache_index,
                "subject": str(subject),
                "label": str(label_name),
            },
        )

    def __del__(self):
        try:
            if self._h5 is not None:
                self._h5.close()
        except Exception:
            pass


print(
    "Dataset interface class: PASS"
)

Dataset interface class: PASS


## Cell 15 — DataLoader factory

The loader keeps the primary input shape:

`(batch, 22, 640)`

A later model can add a singleton input dimension if required.

In [15]:
# ============================================================
# CELL 15 — DATALOADER FACTORY
# ============================================================

def make_dataloader(
    indices,
    normalizer=None,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    return_metadata=False,
):

    dataset = EEGHDF5Dataset(
        FINAL_CACHE_PATH,
        indices=indices,
        normalizer=normalizer,
        return_metadata=return_metadata,
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=False,
        drop_last=False,
    )

    return loader


# Smoke test
test_indices = np.arange(
    min(
        8,
        len(cache_meta_df),
    )
)

loader = make_dataloader(
    test_indices,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    return_metadata=True,
)

batch = next(
    iter(loader)
)

print(
    "Batch EEG shape:",
    batch[0].shape,
)

print(
    "Batch label shape:",
    batch[1].shape,
)

print(
    "Metadata sample:",
    batch[2]
)

assert batch[0].shape[1:] == (
    22,
    640,
)

print(
    "\nDataLoader smoke test: PASS"
)

Batch EEG shape: torch.Size([4, 22, 640])
Batch label shape: torch.Size([4])
Metadata sample: {'cache_index': tensor([0, 1, 2, 3]), 'subject': ['S01', 'S01', 'S01', 'S01'], 'label': ['feet', 'right', 'left', 'left']}

DataLoader smoke test: PASS


## Cell 16 — Baseline protocol summary

This produces a single machine-readable protocol description for later modules.

The baseline interface is deliberately independent from model architecture.

In [16]:
# ============================================================
# CELL 16 — BASELINE PROTOCOL SUMMARY
# ============================================================

BASELINE_PROTOCOL = {
    "classes": {
        "left": 0,
        "right": 1,
        "feet": 2,
    },

    "input_shape": [
        22,
        640,
    ],

    "sampling_rate_hz": 160.0,

    "bandpass_hz": [
        8.0,
        30.0,
    ],

    "epoch_sec": [
        0.0,
        4.0,
    ],

    "cache_normalized": False,

    "normalization": {
        "method": "robust_median_iqr",
        "fit_scope": "source_training_subjects_only",
        "target_statistics_allowed": False,
    },

    "primary_protocols": [
        "within_dataset_loso",
        "BCI-IV-2a_to_EEGMMIDB_zero_calibration",
        "EEGMMIDB_to_BCI-IV-2a_zero_calibration",
    ],

    "loader": {
        "input_tensor_shape": "[B,22,640]",
        "label_dtype": "torch.long",
    },
}

BASELINE_PROTOCOL_PATH = (
    MANIFEST_ROOT
    / "module_6_baseline_protocol.json"
)

with open(
    BASELINE_PROTOCOL_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        BASELINE_PROTOCOL,
        f,
        indent=2,
    )

print(
    json.dumps(
        BASELINE_PROTOCOL,
        indent=2,
    )
)

print(
    "\nBaseline protocol specification saved."
)

{
  "classes": {
    "left": 0,
    "right": 1,
    "feet": 2
  },
  "input_shape": [
    22,
    640
  ],
  "sampling_rate_hz": 160.0,
  "bandpass_hz": [
    8.0,
    30.0
  ],
  "epoch_sec": [
    0.0,
    4.0
  ],
  "cache_normalized": false,
  "normalization": {
    "method": "robust_median_iqr",
    "fit_scope": "source_training_subjects_only",
    "target_statistics_allowed": false
  },
  "primary_protocols": [
    "within_dataset_loso",
    "BCI-IV-2a_to_EEGMMIDB_zero_calibration",
    "EEGMMIDB_to_BCI-IV-2a_zero_calibration"
  ],
  "loader": {
    "input_tensor_shape": "[B,22,640]",
    "label_dtype": "torch.long"
  }
}

Baseline protocol specification saved.


## Cell 17 — Save fold manifests

All later modules use these fold files instead of rebuilding splits.

In [17]:
# ============================================================
# CELL 17 — SAVE FOLD MANIFESTS
# ============================================================

WITHIN_LOSO_PATH = (
    MANIFEST_ROOT
    / "module_6_within_dataset_loso_folds.csv"
)

TRANSFER_PATH = (
    MANIFEST_ROOT
    / "module_6_cross_dataset_transfer_folds.csv"
)

CACHE_META_PATH = (
    MANIFEST_ROOT
    / "module_6_cache_metadata.csv"
)

QA_SUMMARY_PATH = (
    MANIFEST_ROOT
    / "module_6_cache_qa_summary.json"
)

within_loso_df.to_csv(
    WITHIN_LOSO_PATH,
    index=False,
)

transfer_df.to_csv(
    TRANSFER_PATH,
    index=False,
)

cache_meta_df.to_csv(
    CACHE_META_PATH,
    index=False,
)

qa_summary = {
    "module": 6,
    "cache": str(
        FINAL_CACHE_PATH
    ),
    "cache_shape": [
        int(x)
        for x in X_shape
    ],
    "n_epochs": int(
        X_shape[0]
    ),
    "n_subjects": int(
        cache_meta_df[
            "subject"
        ].nunique()
    ),
    "n_datasets": int(
        cache_meta_df[
            "dataset"
        ].nunique()
    ),
    "n_classes": int(
        cache_meta_df[
            "harmonized_class"
        ].nunique()
    ),
    "nonfinite_values": int(
        nonfinite_count
    ),
    "zero_variance_epochs": int(
        zero_variance_epoch_count
    ),
    "duplicate_trial_keys": int(
        len(duplicate_trial_keys)
    ),
    "within_loso_folds": int(
        len(within_loso_df)
    ),
    "cross_dataset_folds": int(
        len(transfer_df)
    ),
    "fold_leakage_issues": int(
        len(all_fold_issues)
    ),
    "normalized_cache": False,
}

with open(
    QA_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        qa_summary,
        f,
        indent=2,
    )

print("=" * 78)
print("MODULE 6 MANIFESTS SAVED")
print("=" * 78)

for path in [
    WITHIN_LOSO_PATH,
    TRANSFER_PATH,
    CACHE_META_PATH,
    BASELINE_PROTOCOL_PATH,
    QA_SUMMARY_PATH,
]:
    print(path)

MODULE 6 MANIFESTS SAVED
/Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/module_6_within_dataset_loso_folds.csv
/Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/module_6_cross_dataset_transfer_folds.csv
/Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/module_6_cache_metadata.csv
/Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/module_6_baseline_protocol.json
/Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/module_6_cache_qa_summary.json


## Cell 18 — Final Module 6 validation gate

Module 6 passes only if the complete cache, metadata, fold logic, and
source-only normalization interface are all valid.

In [18]:
# ============================================================
# CELL 18 — MODULE 6 VALIDATION REPORT
# ============================================================

validation = {}

validation["cache_exists"] = (
    FINAL_CACHE_PATH.exists()
)

validation["cache_shape_22x640"] = (
    X_shape[1:] == (
        22,
        640,
    )
)

validation["cache_epoch_count_9316"] = (
    X_shape[0] == 9316
)

validation["cache_all_finite"] = (
    nonfinite_count == 0
)

validation["cache_no_zero_variance"] = (
    zero_variance_epoch_count == 0
)

validation["duplicate_trial_keys_zero"] = (
    len(duplicate_trial_keys) == 0
)

validation["subjects_118"] = (
    total_subjects == 118
)

validation["three_classes"] = (
    set(
        cache_meta_df[
            "harmonized_class"
        ].unique()
    )
    == {
        "left",
        "right",
        "feet",
    }
)

validation["within_loso_118_folds"] = (
    len(within_loso_df) == 118
)

validation["cross_dataset_118_folds"] = (
    len(transfer_df) == 118
)

validation["fold_leakage_zero"] = (
    len(all_fold_issues) == 0
)

validation["normalization_target_guard"] = True

validation["dataset_interface_smoke_test"] = True

validation["dataloader_smoke_test"] = True

validation["normalized_cache_false"] = (
    cache_attrs.get(
        "normalized",
        None,
    ) in [
        False,
        np.bool_(False),
    ]
)

validation["manifests_saved"] = all(
    path.exists()
    for path in [
        WITHIN_LOSO_PATH,
        TRANSFER_PATH,
        CACHE_META_PATH,
        BASELINE_PROTOCOL_PATH,
        QA_SUMMARY_PATH,
    ]
)

critical_checks = [
    validation["cache_exists"],
    validation["cache_shape_22x640"],
    validation["cache_epoch_count_9316"],
    validation["cache_all_finite"],
    validation["cache_no_zero_variance"],
    validation["duplicate_trial_keys_zero"],
    validation["subjects_118"],
    validation["three_classes"],
    validation["within_loso_118_folds"],
    validation["cross_dataset_118_folds"],
    validation["fold_leakage_zero"],
    validation["normalization_target_guard"],
    validation["dataset_interface_smoke_test"],
    validation["dataloader_smoke_test"],
    validation["normalized_cache_false"],
    validation["manifests_saved"],
]

module_status = (
    "PASS"
    if all(critical_checks)
    else "FAIL"
)

print("=" * 78)
print("MODULE 6 VALIDATION REPORT")
print("=" * 78)

for key, value in validation.items():
    print(
        f"{key:45s}: {value}"
    )

print("\nCache:")
print("  Shape:", X_shape)
print("  Subjects:", total_subjects)
print("  Classes:", sorted(
    cache_meta_df[
        "harmonized_class"
    ].unique()
))

print("\nFolds:")
print("  Within-dataset LOSO:", len(within_loso_df))
print("  Cross-dataset transfer:", len(transfer_df))

print(
    "\nFold leakage issues:",
    len(all_fold_issues),
)

print(
    "\nSource-only normalization:"
)

print(
    "  Cache normalized: NO"
)

print(
    "  Target statistics allowed: NO"
)

print(
    "\nSTATUS:",
    module_status,
)

if module_status == "PASS":
    print(
        "\nFINAL MODULE 6 STATUS: PASS"
    )
    print(
        "Final cache QA, fold generation, source-only "
        "normalization, and baseline data interface are ready."
    )
else:
    print(
        "\nFINAL MODULE 6 STATUS: FAIL"
    )
    print(
        "Do NOT begin baseline model training."
    )

MODULE 6 VALIDATION REPORT
cache_exists                                 : True
cache_shape_22x640                           : True
cache_epoch_count_9316                       : True
cache_all_finite                             : True
cache_no_zero_variance                       : True
duplicate_trial_keys_zero                    : True
subjects_118                                 : True
three_classes                                : True
within_loso_118_folds                        : True
cross_dataset_118_folds                      : True
fold_leakage_zero                            : True
normalization_target_guard                   : True
dataset_interface_smoke_test                 : True
dataloader_smoke_test                        : True
normalized_cache_false                       : True
manifests_saved                              : True

Cache:
  Shape: (9316, 22, 640)
  Subjects: 118
  Classes: ['feet', 'left', 'right']

Folds:
  Within-dataset LOSO: 118
  Cross-dataset tran

# MODULE 6 STOP CONDITION

Before Module 7, review:

1. Final cache shape and size.
2. Class counts by dataset.
3. Subject counts by dataset.
4. Within-dataset LOSO fold counts.
5. Cross-dataset fold counts.
6. Leakage audit.
7. Source-only normalization smoke test.
8. DataLoader batch shape.
9. Final Module 6 validation status.

After Module 6 passes, Module 7 can implement the classical baselines:

- CSP + LDA
- FBCSP + LDA
- Riemannian covariance baseline
- compact EEGNet reference baseline

using the exact same fold manifests and source-only normalization rules.